In [9]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv

In [10]:
load_dotenv()

conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA'),
    role=os.getenv('SNOWFLAKE_ROLE')
)

# Analytical Marts

> Pre-joining everything into one wide, flat table optimized for specific analytical purposes.

These marts are **ready-to-query** tables designed for BI tools, dashboards, and ad-hoc analysis.

In [11]:
query = """SELECT order_id, order_ts, order_status, carrier, warehouse_origin
FROM sales_staging.stg_orders
LIMIT 20;"""

df = pd.read_sql(query, conn)
df.columns = df.columns.str.lower()

print(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_44544\600662657.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


      order_id            order_ts order_status         carrier  \
0   ORD-000001 2024-08-17 11:59:43    Delivered      BudgetPost   
1   ORD-000002 2024-12-01 21:24:49    Delivered      NationWide   
2   ORD-000003 2024-04-02 19:19:46    Delivered  PrimeLogistics   
3   ORD-000004 2023-02-01 19:27:34    Delivered  PrimeLogistics   
4   ORD-000005 2023-12-22 17:39:39    Delivered        FastShip   
5   ORD-000006 2024-07-07 22:16:35    Delivered      NationWide   
6   ORD-000007 2023-06-15 02:35:36    Delivered  PrimeLogistics   
7   ORD-000008 2024-10-26 13:20:34    Delivered  PrimeLogistics   
8   ORD-000009 2024-07-06 07:06:13    Delivered      BudgetPost   
9   ORD-000010 2024-06-24 07:26:19    Delivered      BudgetPost   
10  ORD-000011 2024-10-22 13:19:14    Delivered      NationWide   
11  ORD-000012 2024-06-07 19:28:33    Delivered        FastShip   
12  ORD-000013 2024-12-04 11:08:26    Delivered      NationWide   
13  ORD-000014 2024-03-18 18:40:06    Delivered        FastShi

In [12]:
query = """-- Verify: does this produce exactly one row per order?
SELECT
    order_id,
    COUNT(*)         AS line_item_count,
    SUM(quantity)    AS total_units,
    SUM(item_total)  AS items_revenue,
    SUM(item_profit) AS total_profit,
    AVG(margin_pct)  AS avg_margin_pct
FROM sales_staging.stg_order_items
GROUP BY order_id
ORDER BY line_item_count DESC
LIMIT 10;"""

df = pd.read_sql(query, conn)
df.columns = df.columns.str.lower()

print(df)

     order_id  line_item_count  total_units  items_revenue  total_profit  \
0  ORD-000034                5            5         266.59         96.64   
1  ORD-000049                5            7        1211.65        252.82   
2  ORD-000006                5            6         375.47        103.77   
3  ORD-000190                5            8        1231.74        196.44   
4  ORD-000064                5            6        1469.38         21.50   
5  ORD-000096                5            5         661.01        119.60   
6  ORD-000100                5            9         573.67        214.88   
7  ORD-000025                5           10        1952.33        497.66   
8  ORD-000072                5            5         672.22        189.51   
9  ORD-000013                5            9        1283.63        501.53   

   avg_margin_pct  
0          33.078  
1          29.964  
2          32.864  
3          29.634  
4          12.706  
5          19.192  
6          31.306  
7  

C:\Users\frase\AppData\Local\Temp\ipykernel_44544\795876496.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [13]:
query = """
SELECT
    order_id,
    COUNT(*)                   AS return_count,
    SUM(refund_amount)         AS total_refund,
    SUM(total_return_cost)     AS total_return_cost,
    MAX(return_reason)         AS primary_return_reason
FROM sales_staging.stg_returns
GROUP BY order_id
LIMIT 10;"""

df = pd.read_sql(query, conn)
df.columns = df.columns.str.lower()

print(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_44544\1625793337.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


     order_id  return_count  total_refund  total_return_cost  \
0  ORD-000105             1        638.36             638.36   
1  ORD-000184             1        127.04             127.04   
2  ORD-000214             2       1959.74            2262.15   
3  ORD-000404             1         37.72              37.72   
4  ORD-000479             1         28.74              28.74   
5  ORD-000926             1        138.38             138.38   
6  ORD-000198             1        889.58             889.58   
7  ORD-000480             2        657.57             657.57   
8  ORD-000576             1        147.49             147.49   
9  ORD-000707             1        146.68             146.68   

        primary_return_reason  
0  Quality Below Expectations  
1            Not as Described  
2               Late Delivery  
3               Late Delivery  
4  Quality Below Expectations  
5             Too Small/Large  
6           Defective/Damaged  
7               Late Delivery  
8      

In [14]:
query = """SELECT *
FROM (
    SELECT
        order_id,
        rating,
        delivery_rating,
        product_rating,
        was_delivery_late,
        days_late,
        ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_date DESC) AS rn
    FROM sales_raw."08_CUSTOMER_REVIEWS"
)
WHERE rn = 1
LIMIT 10"""

df = pd.read_sql(query, conn)
df.columns = df.columns.str.lower()

print(df)

     order_id  rating  delivery_rating  product_rating  was_delivery_late  \
0  ORD-002242       4                5               4                  0   
1  ORD-010563       5                5               3                  0   
2  ORD-004727       3                4               3                  0   
3  ORD-008302       3                2               3                  1   
4  ORD-009707       4                4               2                  0   
5  ORD-010972       5                5               4                  0   
6  ORD-006276       4                4               4                  0   
7  ORD-011546       5                5               4                  0   
8  ORD-000559       3                4               4                  0   
9  ORD-006127       4                4               3                  0   

   days_late  rn  
0          0   1  
1          0   1  
2          0   1  
3          1   1  
4          0   1  
5          0   1  
6          0   1  


C:\Users\frase\AppData\Local\Temp\ipykernel_44544\1640478072.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


---

## 📦 Mart: `mart_order_complete`

**Grain:** One row per order (order-level analysis)

### Joins Included

| Source | Fields Added |
|--------|--------------|
| `stg_orders` | Base order data, lifecycle metrics, delivery performance |
| `05_sellers` | Seller name, quality score, fulfillment type |
| `04_customers` | Acquisition channel, segment, lifetime value |
| `stg_order_items` | Line item count, revenue, profit, primary category |
| `stg_returns` | Return count, refund amount, return reason |
| `08_customer_reviews` | Rating, delivery rating, product rating |
| `11_support_tickets` | Ticket count, primary issue type |

In [15]:
query = """
CREATE OR REPLACE TABLE sales_marts.mart_order_complete AS

SELECT
    -- === ORDER BASE (from staging) ===
    o.order_id,
    o.customer_id,
    o.seller_id,
    o.order_ts,
    o.order_month,
    o.order_year,
    o.order_quarter,
    o.order_month_num,
    o.order_dow,
    o.order_hour,
    o.order_status,
    o.carrier,
    o.warehouse_origin,
    o.customer_region,
    o.customer_city,
    o.order_channel,
    o.payment_method,
    o.is_weekend_order,
    o.order_total,
    o.shipping_cost,
    o.total_weight_kg,

    -- === LIFECYCLE METRICS (from staging) ===
    o.hours_to_approve,
    o.hours_to_ship,
    o.hours_in_transit,
    o.hours_total_lifecycle,
    o.is_late_delivery,
    o.days_late,

    -- === DATA QUALITY FLAGS ===
    o.flag_missing_customer,
    o.flag_impossible_date,
    o.flag_missing_delivery_date,

    -- === SELLER CONTEXT (from raw, small table) ===
    s.seller_name,
    s.warehouse_location,
    s.quality_score          AS seller_quality_score,
    s.fulfillment_type,
    s.avg_processing_hours   AS seller_avg_processing_hrs,
    s.is_verified            AS seller_is_verified,

    -- === CUSTOMER CONTEXT (from raw) ===
    c.acquisition_channel,
    c.customer_segment,
    c.lifetime_value_estimate,
    c.signup_date            AS customer_signup_date,
    c.has_app                AS customer_has_app,

    -- === ITEM AGGREGATES (one row per order) ===
    COALESCE(items.line_item_count, 0)  AS line_item_count,
    COALESCE(items.total_units, 0)      AS total_units,
    COALESCE(items.items_revenue, 0)    AS items_revenue,
    COALESCE(items.total_profit, 0)     AS total_profit,
    items.avg_margin_pct,
    items.primary_category,

    -- === RETURN INFO (aggregated to order grain) ===
    CASE WHEN ret.order_id IS NOT NULL THEN 1 ELSE 0 END  AS has_return,
    COALESCE(ret.return_count, 0)        AS return_count,
    COALESCE(ret.total_refund, 0)        AS total_refund,
    ret.primary_return_reason,

    -- === REVIEW INFO (deduplicated to one per order) ===
    rev.rating                AS review_rating,
    rev.delivery_rating       AS review_delivery_rating,
    rev.product_rating        AS review_product_rating,
    rev.was_delivery_late     AS review_flagged_late,

    -- === SUPPORT TICKET INFO ===
    CASE WHEN tkt.order_id IS NOT NULL THEN 1 ELSE 0 END  AS has_support_ticket,
    COALESCE(tkt.ticket_count, 0)        AS ticket_count,
    tkt.primary_issue_type

FROM sales_staging.stg_orders o

-- Seller context
LEFT JOIN sales_raw."05_SELLERS" s
    ON o.seller_id = s.seller_id

-- Customer context
LEFT JOIN sales_raw."04_CUSTOMERS" c
    ON o.customer_id = c.customer_id

-- Item aggregates (subquery ensures one row per order)
LEFT JOIN (
    SELECT
        order_id,
        COUNT(*)                          AS line_item_count,
        SUM(quantity)                     AS total_units,
        SUM(item_total)                   AS items_revenue,
        SUM(item_profit)                  AS total_profit,
        ROUND(AVG(margin_pct), 2)         AS avg_margin_pct,
        MAX_BY(category, item_total)      AS primary_category
    FROM sales_staging.stg_order_items
    GROUP BY order_id
) items ON o.order_id = items.order_id

-- Return aggregates (subquery ensures one row per order)
LEFT JOIN (
    SELECT
        order_id,
        COUNT(*)                          AS return_count,
        SUM(refund_amount)                AS total_refund,
        MAX_BY(return_reason, refund_amount) AS primary_return_reason
    FROM sales_staging.stg_returns
    GROUP BY order_id
) ret ON o.order_id = ret.order_id

-- Review (deduplicated: most recent review per order)
LEFT JOIN (
    SELECT order_id, rating, delivery_rating, product_rating, was_delivery_late
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_date DESC) AS rn
        FROM sales_raw."08_CUSTOMER_REVIEWS"
    )
    WHERE rn = 1
) rev ON o.order_id = rev.order_id

-- Support ticket aggregates
LEFT JOIN (
    SELECT
        order_id,
        COUNT(*)                              AS ticket_count,
        MAX_BY(issue_type, created_date)      AS primary_issue_type
    FROM sales_raw."11_SUPPORT_TICKETS"
    GROUP BY order_id
) tkt ON o.order_id = tkt.order_id;
"""

cursor = conn.cursor()
cursor.execute(query)
print("✓ Table SALES_MARTS.MART_ORDER_COMPLETE created successfully")
print(f"  Rows affected: {cursor.rowcount}")

✓ Table SALES_MARTS.MART_ORDER_COMPLETE created successfully
  Rows affected: 1


In [16]:
# Verification: Row counts and data quality checks
cursor = conn.cursor()

# 1. Row count comparison (should match)
cursor.execute("SELECT COUNT(*) AS cnt FROM sales_staging.stg_orders")
stg_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) AS cnt FROM sales_marts.mart_order_complete")
mart_count = cursor.fetchone()[0]

print(f"stg_orders count:        {stg_count}")
print(f"mart_order_complete count: {mart_count}")
print(f"Match: {'✓' if stg_count == mart_count else '✗ MISMATCH - check for duplicates!'}\n")

# 2. Spot check metrics
query = """
SELECT
    COUNT(*)                                          AS total_orders,
    SUM(has_return)                                   AS orders_with_returns,
    ROUND(100.0 * SUM(has_return) / COUNT(*), 2)      AS return_rate_pct,
    SUM(has_support_ticket)                           AS orders_with_tickets,
    COUNT(review_rating)                              AS orders_with_reviews,
    ROUND(AVG(hours_total_lifecycle), 1)              AS avg_lifecycle_hours,
    ROUND(SUM(total_profit), 2)                       AS total_gross_profit
FROM sales_marts.mart_order_complete
"""
df = pd.read_sql(query, conn)
df.columns = df.columns.str.lower()
print("Spot Check Metrics:")
print(df.to_string(index=False))

# 3. Duplicate check (should return 0 rows)
query = """
SELECT order_id, COUNT(*) AS cnt
FROM sales_marts.mart_order_complete
GROUP BY order_id
HAVING COUNT(*) > 1
"""
df_dups = pd.read_sql(query, conn)
print(f"\nDuplicate orders found: {len(df_dups)}")
if len(df_dups) > 0:
    print(df_dups)

stg_orders count:        15000
mart_order_complete count: 15000
Match: ✓

Spot Check Metrics:
 total_orders  orders_with_returns  return_rate_pct  orders_with_tickets  orders_with_reviews  avg_lifecycle_hours  total_gross_profit
        15000                 2872            19.15                 2250                 8686                142.2          1939411.28


C:\Users\frase\AppData\Local\Temp\ipykernel_44544\1448321506.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
C:\Users\frase\AppData\Local\Temp\ipykernel_44544\1448321506.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dups = pd.read_sql(query, conn)



Duplicate orders found: 0


---

## 🛒 Mart: `mart_order_items_detail`

**Grain:** One row per order item (line-item level analysis)

### Questions This Mart Answers

| Analysis Area | Example Questions |
|--------------|-------------------|
| Margins | Which product-category + warehouse combos have the worst margins? |
| Discounting | Which subcategories get discounted the most? Does it hurt profitability? |
| Returns | What's the return rate for fragile items shipped via BudgetPost vs others? |
| Seller Performance | Which seller is selling high-volume, low-margin products not worth fulfillment cost? |

In [17]:
query = """CREATE OR REPLACE TABLE sales_marts.mart_order_items_detail AS

SELECT
    -- === ITEM FIELDS ===
    oi.order_item_id,
    oi.order_id,
    oi.product_id,
    oi.seller_id,
    oi.quantity,
    oi.unit_price,
    oi.list_price,
    oi.cost_price,
    oi.discount_pct,
    oi.item_total,
    oi.item_profit,
    oi.margin_pct,
    oi.category,
    oi.subcategory,
    oi.is_fragile,
    oi.is_hazardous,
    oi.was_discounted,

    -- === ORDER CONTEXT (joined down) ===
    o.order_ts,
    o.order_month,
    o.order_year,
    o.order_quarter,
    o.carrier,
    o.warehouse_origin,
    o.customer_region,
    o.is_weekend_order,
    o.order_channel,
    o.hours_to_ship,
    o.hours_in_transit,
    o.hours_total_lifecycle,
    o.is_late_delivery,

    -- === SELLER CONTEXT ===
    s.fulfillment_type,
    s.quality_score       AS seller_quality_score,
    s.warehouse_location,

    -- === RETURN STATUS (item-level, not aggregated) ===
    CASE WHEN r.return_id IS NOT NULL THEN 1 ELSE 0 END AS was_returned,
    r.return_reason,
    r.return_condition,
    r.refund_amount,
    r.total_return_cost

FROM sales_staging.stg_order_items oi

LEFT JOIN sales_staging.stg_orders o
    ON oi.order_id = o.order_id

LEFT JOIN sales_raw."05_SELLERS" s
    ON oi.seller_id = s.seller_id

-- Item-level returns (this join is safe because returns are per item)
LEFT JOIN sales_staging.stg_returns r
    ON oi.order_item_id = r.order_item_id;"""

cursor = conn.cursor()
cursor.execute(query)
print("✓ Table SALES_MARTS.MART_ORDER_ITEMS_DETAIL created successfully")
print(f"  Rows affected: {cursor.rowcount}")

✓ Table SALES_MARTS.MART_ORDER_ITEMS_DETAIL created successfully
  Rows affected: 1


In [18]:
# Verification: Row counts and duplicate check
cursor = conn.cursor()

# 1. Row count comparison (should match)
cursor.execute("SELECT COUNT(*) AS cnt FROM sales_staging.stg_order_items")
stg_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) AS cnt FROM sales_marts.mart_order_items_detail")
mart_count = cursor.fetchone()[0]

print(f"stg_order_items count:        {stg_count}")
print(f"mart_order_items_detail count: {mart_count}")
print(f"Match: {'✓' if stg_count == mart_count else '✗ MISMATCH - check for duplicates!'}\n")

# 2. Duplicate check (should return 0 rows)
query = """
SELECT order_item_id, COUNT(*) AS cnt
FROM sales_marts.mart_order_items_detail
GROUP BY order_item_id
HAVING COUNT(*) > 1
"""
df_dups = pd.read_sql(query, conn)
print(f"Duplicate order_items found: {len(df_dups)}")
if len(df_dups) > 0:
    print(df_dups)

stg_order_items count:        29668
mart_order_items_detail count: 29668
Match: ✓



C:\Users\frase\AppData\Local\Temp\ipykernel_44544\1467099530.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dups = pd.read_sql(query, conn)


Duplicate order_items found: 0


---

## 👤 Mart: `mart_customer_summary`

**Grain:** One row per customer (customer-level analysis)

### Use Cases

| Analysis Type | Description |
|--------------|-------------|
| Customer Segmentation | Group customers by behavior, spend, and engagement |
| Cohort Analysis | Track customer groups by signup date or first order |
| Lifetime Value | Analyze total spend, order frequency, and retention |
| Churn Risk | Identify customers with poor delivery experience or high return rates |

In [19]:
query = """CREATE OR REPLACE TABLE sales_marts.mart_customer_summary AS

SELECT
    c.customer_id,
    c.region,
    c.city,
    c.acquisition_channel,
    c.customer_segment,
    c.signup_date,
    c.lifetime_value_estimate,
    c.has_app,

    -- === ORDER HISTORY ===
    COUNT(o.order_id)                                  AS total_orders,
    MIN(o.order_ts)                                    AS first_order_date,
    MAX(o.order_ts)                                    AS last_order_date,
    DATEDIFF('day', MIN(o.order_ts), MAX(o.order_ts))  AS customer_lifespan_days,
    SUM(o.order_total)                                 AS total_spend,
    AVG(o.order_total)                                 AS avg_order_value,

    -- === EXPERIENCE METRICS ===
    AVG(o.hours_total_lifecycle)                        AS avg_delivery_hours,
    SUM(o.is_late_delivery)                            AS late_delivery_count,
    ROUND(100.0 * SUM(o.is_late_delivery) /
        NULLIF(COUNT(o.order_id), 0), 2)               AS late_delivery_pct,

    -- === RETURN BEHAVIOR ===
    SUM(o.has_return)                                  AS total_returns,
    SUM(o.total_refund)                                AS total_refund_amount,
    ROUND(100.0 * SUM(o.has_return) /
        NULLIF(COUNT(o.order_id), 0), 2)               AS return_rate_pct,

    -- === REVIEW BEHAVIOR ===
    AVG(o.review_rating)                               AS avg_rating_given,
    COUNT(o.review_rating)                             AS reviews_given

FROM sales_raw."04_CUSTOMERS" c
LEFT JOIN sales_marts.mart_order_complete o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id, c.region, c.city, c.acquisition_channel,
    c.customer_segment, c.signup_date, c.lifetime_value_estimate, c.has_app;"""

cursor = conn.cursor()
cursor.execute(query)
print("✓ Table SALES_MARTS.MART_CUSTOMER_SUMMARY created successfully")
print(f"  Rows affected: {cursor.rowcount}")

✓ Table SALES_MARTS.MART_CUSTOMER_SUMMARY created successfully
  Rows affected: 1
